# 03 — Data Cleaning

## Sales Order, Backlog & Inventory Analytics

### Objective

This notebook cleans the confirmed data-quality issues identified during the Data Quality Audit.

The cleaning process will focus on:

- Converting validated date columns to proper datetime types
- Removing exact duplicate inventory transactions
- Handling missing customer credit tiers
- Handling missing product subcategories
- Handling missing supplier quality ratings
- Treating invalid shipment chronology
- Preserving structurally valid missing values in open purchase orders
- Performing post-cleaning validation
- Exporting clean datasets for downstream SQL and analytical workflows

Only issues confirmed during the Data Quality Audit will be modified.
Valid operational observations will be preserved.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)
print("\nData Cleaning environment initialized successfully.")

Pandas version : 3.0.6
NumPy version  : 2.5.3

Data Cleaning environment initialized successfully.


## Step 2 — Define Project Paths

Define the project directories used during the cleaning process.

The raw source files will remain unchanged. Cleaned datasets will later be exported to a separate processed-data directory to preserve the original data.

In [2]:
# Define project paths

project_path = Path.cwd().parent

raw_data_path = project_path / "data" / "raw"
processed_data_path = project_path / "data" / "processed"


print("Project Path        :", project_path)
print("Raw Data Path       :", raw_data_path)
print("Processed Data Path :", processed_data_path)

print("\nRaw Data Folder Exists       :", raw_data_path.exists())
print("Processed Data Folder Exists :", processed_data_path.exists())

Project Path        : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Raw Data Path       : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\raw
Processed Data Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\processed

Raw Data Folder Exists       : True
Processed Data Folder Exists : True


## Step 3 — Load Raw Datasets

Load all source datasets from the raw-data directory.

The original raw files are loaded into memory without modification. All cleaning operations will be performed on separate DataFrame copies to preserve the source data.

In [3]:
# Check available raw CSV files

raw_files = sorted(raw_data_path.glob("*.csv"))

print(f"CSV Files Found : {len(raw_files)}\n")

for file in raw_files:
    print(file.name)

CSV Files Found : 10

customers.csv
inventory.csv
inventory_transactions.csv
products.csv
purchase_orders.csv
sales_order_items.csv
sales_orders.csv
shipments.csv
suppliers.csv
warehouses.csv


## Step 4 — Load Raw Datasets into DataFrames

Load all 10 raw CSV files into separate Pandas DataFrames.

The raw source files remain unchanged on disk. Cleaning operations will be performed on DataFrame copies created in later steps.

In [4]:
# Load all raw datasets

customers = pd.read_csv(raw_data_path / "customers.csv")
products = pd.read_csv(raw_data_path / "products.csv")
sales_orders = pd.read_csv(raw_data_path / "sales_orders.csv")
sales_order_items = pd.read_csv(raw_data_path / "sales_order_items.csv")
inventory = pd.read_csv(raw_data_path / "inventory.csv")
inventory_transactions = pd.read_csv(
    raw_data_path / "inventory_transactions.csv"
)
warehouses = pd.read_csv(raw_data_path / "warehouses.csv")
shipments = pd.read_csv(raw_data_path / "shipments.csv")
suppliers = pd.read_csv(raw_data_path / "suppliers.csv")
purchase_orders = pd.read_csv(raw_data_path / "purchase_orders.csv")


print("Raw datasets loaded successfully.\n")

print(f"Customers              : {customers.shape}")
print(f"Products               : {products.shape}")
print(f"Sales Orders           : {sales_orders.shape}")
print(f"Sales Order Items      : {sales_order_items.shape}")
print(f"Inventory              : {inventory.shape}")
print(f"Inventory Transactions : {inventory_transactions.shape}")
print(f"Warehouses             : {warehouses.shape}")
print(f"Shipments              : {shipments.shape}")
print(f"Suppliers              : {suppliers.shape}")
print(f"Purchase Orders        : {purchase_orders.shape}")

Raw datasets loaded successfully.

Customers              : (2500, 6)
Products               : (500, 10)
Sales Orders           : (25000, 14)
Sales Order Items      : (61397, 16)
Inventory              : (3000, 11)
Inventory Transactions : (110045, 6)
Warehouses             : (6, 5)
Shipments              : (24798, 10)
Suppliers              : (100, 7)
Purchase Orders        : (8500, 14)


## Step 5 — Create Working Copies

Create independent working copies of all raw DataFrames before applying any cleaning transformations.

This preserves the originally loaded datasets in memory and allows the cleaned versions to be compared against the raw data during post-cleaning validation.

In [5]:
# Create working copies for data cleaning

customers_clean = customers.copy()
products_clean = products.copy()
sales_orders_clean = sales_orders.copy()
sales_order_items_clean = sales_order_items.copy()
inventory_clean = inventory.copy()
inventory_transactions_clean = inventory_transactions.copy()
warehouses_clean = warehouses.copy()
shipments_clean = shipments.copy()
suppliers_clean = suppliers.copy()
purchase_orders_clean = purchase_orders.copy()


print("Working copies created successfully.\n")

print(f"Customers              : {customers_clean.shape}")
print(f"Products               : {products_clean.shape}")
print(f"Sales Orders           : {sales_orders_clean.shape}")
print(f"Sales Order Items      : {sales_order_items_clean.shape}")
print(f"Inventory              : {inventory_clean.shape}")
print(f"Inventory Transactions : {inventory_transactions_clean.shape}")
print(f"Warehouses             : {warehouses_clean.shape}")
print(f"Shipments              : {shipments_clean.shape}")
print(f"Suppliers              : {suppliers_clean.shape}")
print(f"Purchase Orders        : {purchase_orders_clean.shape}")

Working copies created successfully.

Customers              : (2500, 6)
Products               : (500, 10)
Sales Orders           : (25000, 14)
Sales Order Items      : (61397, 16)
Inventory              : (3000, 11)
Inventory Transactions : (110045, 6)
Warehouses             : (6, 5)
Shipments              : (24798, 10)
Suppliers              : (100, 7)
Purchase Orders        : (8500, 14)


## Step 6 — Convert Date Columns to Datetime

Convert all validated date fields from string/object format to Pandas datetime format.

The Data Quality Audit confirmed that all non-null date values are valid and parseable. Therefore, these fields can be safely converted without introducing parsing errors.

Structurally valid missing dates in open Purchase Orders are preserved as `NaT`.

In [6]:
# Define date columns for each dataset

date_columns = {
    "Customers": (
        customers_clean,
        ["customer_since"]
    ),

    "Sales Orders": (
        sales_orders_clean,
        [
            "order_date",
            "requested_delivery_date"
        ]
    ),

    "Sales Order Items": (
        sales_order_items_clean,
        ["order_date"]
    ),

    "Inventory Transactions": (
        inventory_transactions_clean,
        ["transaction_date"]
    ),

    "Shipments": (
        shipments_clean,
        [
            "shipment_date",
            "requested_delivery_date",
            "actual_delivery_date"
        ]
    ),

    "Purchase Orders": (
        purchase_orders_clean,
        [
            "po_date",
            "expected_date",
            "actual_receipt_date"
        ]
    )
}


# Convert date columns

for table_name, (df, columns) in date_columns.items():

    for column in columns:

        df[column] = pd.to_datetime(
            df[column],
            errors="raise"
        )


print("Date conversion completed successfully.\n")


# Validate resulting data types

for table_name, (df, columns) in date_columns.items():

    print(f"{table_name}:")

    for column in columns:
        print(
            f"  {column:<28} "
            f"{df[column].dtype}"
        )

    print()

Date conversion completed successfully.

Customers:
  customer_since               datetime64[us]

Sales Orders:
  order_date                   datetime64[us]
  requested_delivery_date      datetime64[us]

Sales Order Items:
  order_date                   datetime64[us]

Inventory Transactions:
  transaction_date             datetime64[us]

Shipments:
  shipment_date                datetime64[us]
  requested_delivery_date      datetime64[us]
  actual_delivery_date         datetime64[us]

Purchase Orders:
  po_date                      datetime64[us]
  expected_date                datetime64[us]
  actual_receipt_date          datetime64[us]



## Step 7 — Remove Exact Duplicate Inventory Transactions

The Data Quality Audit identified 45 exact duplicate records in the Inventory Transactions dataset.

Further investigation confirmed that:

- The duplicated transaction IDs correspond to exact duplicate rows.
- No conflicting records exist for the duplicated IDs.
- One copy of each duplicated transaction can therefore be safely removed.

The expected row count after deduplication is 110,000.

In [7]:
# Remove exact duplicate inventory transactions

rows_before = len(inventory_transactions_clean)

duplicates_before = (
    inventory_transactions_clean
    .duplicated()
    .sum()
)


inventory_transactions_clean = (
    inventory_transactions_clean
    .drop_duplicates()
    .reset_index(drop=True)
)


rows_after = len(inventory_transactions_clean)

duplicates_after = (
    inventory_transactions_clean
    .duplicated()
    .sum()
)


duplicate_ids_after = (
    inventory_transactions_clean[
        "transaction_id"
    ]
    .duplicated()
    .sum()
)


print(f"Rows Before Cleaning       : {rows_before:,}")
print(f"Exact Duplicates Found     : {duplicates_before:,}")
print(f"Rows Removed               : {rows_before - rows_after:,}")
print(f"Rows After Cleaning        : {rows_after:,}")

print(
    f"Exact Duplicates Remaining : "
    f"{duplicates_after:,}"
)

print(
    f"Duplicate IDs Remaining    : "
    f"{duplicate_ids_after:,}"
)

Rows Before Cleaning       : 110,045
Exact Duplicates Found     : 45
Rows Removed               : 45
Rows After Cleaning        : 110,000
Exact Duplicates Remaining : 0
Duplicate IDs Remaining    : 0


## Step 8 — Evaluate Missing Customer Credit Tier Treatment

Before filling missing customer credit tiers, evaluate the distribution of existing credit tiers across customer segments.

The goal is to determine whether a defensible category can be inferred from existing customer attributes or whether the missing values should be preserved using an explicit `Unknown` category.

No values are modified in this step.

In [8]:
# Analyze credit tier distribution by customer segment

credit_tier_distribution = pd.crosstab(
    customers_clean["customer_segment"],
    customers_clean["credit_tier"],
    normalize="index"
).mul(100).round(2)


print("Credit Tier Distribution by Customer Segment (%):\n")

print(
    credit_tier_distribution
    .to_string()
)


print("\nCustomers With Missing Credit Tier:\n")

print(
    customers_clean.loc[
        customers_clean["credit_tier"].isna(),
        [
            "customer_id",
            "customer_segment",
            "region",
            "customer_since"
        ]
    ]
    .to_string(index=False)
)

Credit Tier Distribution by Customer Segment (%):

credit_tier           A      B      C
customer_segment                     
Distributor       37.25  46.50  16.25
Enterprise        43.30  42.34  14.35
Retail            40.76  41.19  18.05
SMB               42.44  41.54  16.03

Customers With Missing Credit Tier:

customer_id customer_segment  region customer_since
  CUST00072      Distributor Central     2018-04-14
  CUST00102           Retail    West     2023-08-10
  CUST00103              SMB    West     2021-03-24
  CUST00391       Enterprise    East     2024-06-20
  CUST00620           Retail Central     2019-06-19
  CUST00899              SMB   North     2021-02-03
  CUST01042       Enterprise   South     2024-06-18
  CUST01053           Retail Central     2022-05-29
  CUST01063              SMB   South     2019-07-05
  CUST01167              SMB    West     2021-07-29
  CUST01266              SMB   North     2021-01-25
  CUST01330      Distributor   North     2019-08-20
  CUST0

## Step 9 — Handle Missing Customer Credit Tier

The customer credit-tier distribution was evaluated across customer segments.

No customer segment showed a sufficiently strong or deterministic relationship with a specific credit tier. Therefore, imputing A, B, or C based on customer segment would introduce unsupported assumptions.

The 20 missing credit-tier values are assigned the explicit category `Unknown`.

This preserves all customer records while clearly distinguishing unavailable credit information from known credit tiers.


In [9]:
# Handle missing customer credit tier

missing_before = (
    customers_clean["credit_tier"]
    .isna()
    .sum()
)


customers_clean["credit_tier"] = (
    customers_clean["credit_tier"]
    .fillna("Unknown")
)


missing_after = (
    customers_clean["credit_tier"]
    .isna()
    .sum()
)


unknown_count = (
    customers_clean["credit_tier"]
    .eq("Unknown")
    .sum()
)


print(f"Missing Before Cleaning : {missing_before:,}")
print(f"Values Assigned Unknown : {unknown_count:,}")
print(f"Missing After Cleaning  : {missing_after:,}")


print("\nCredit Tier Distribution After Cleaning:")

print(
    customers_clean["credit_tier"]
    .value_counts(dropna=False)
    .to_string()
)

Missing Before Cleaning : 20
Values Assigned Unknown : 20
Missing After Cleaning  : 0

Credit Tier Distribution After Cleaning:
credit_tier
B          1048
A          1022
C           410
Unknown      20


## Step 10 — Evaluate Missing Product Subcategories

Investigate products with missing subcategories before applying any treatment.

Product names and existing category-subcategory mappings are reviewed to determine whether the missing subcategories can be inferred reliably from consistent product naming patterns.

No values are modified in this step.

In [10]:
# Identify products with missing subcategories

missing_subcategory_products = products_clean.loc[
    products_clean["subcategory"].isna(),
    [
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "active_flag"
    ]
]


print("Products With Missing Subcategory:\n")

print(
    missing_subcategory_products
    .to_string(index=False)
)


# Extract the descriptive product-name prefix
# Example: "Cookware Product 0184" -> "Cookware"

products_clean["product_name_group"] = (
    products_clean["product_name"]
    .str.replace(
        r"\s+Product\s+\d+$",
        "",
        regex=True
    )
)


print("\nExisting Mapping for Affected Product Name Groups:\n")

affected_name_groups = (
    products_clean.loc[
        products_clean["subcategory"].isna(),
        "product_name_group"
    ]
    .unique()
)


mapping_check = (
    products_clean.loc[
        products_clean["product_name_group"].isin(
            affected_name_groups
        )
        & products_clean["subcategory"].notna(),
        [
            "product_name_group",
            "category",
            "subcategory"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "product_name_group",
            "category",
            "subcategory"
        ]
    )
)


print(
    mapping_check
    .to_string(index=False)
)

Products With Missing Subcategory:

product_id          product_name        category subcategory active_flag
   SKU0184 Cookware Product 0184  Home & Kitchen         NaN           Y
   SKU0281  Storage Product 0281  Home & Kitchen         NaN           Y
   SKU0424 Printing Product 0424 Office Supplies         NaN           Y

Existing Mapping for Affected Product Name Groups:

product_name_group        category subcategory
          Cookware  Home & Kitchen    Cookware
          Printing Office Supplies    Printing
           Storage  Home & Kitchen     Storage


## Step 11 — Handle Missing Product Subcategories

The three missing product subcategories were investigated using existing product naming patterns and category-subcategory mappings.

Each affected product name group maps consistently to exactly one existing subcategory within its corresponding category:

- Cookware → Cookware
- Storage → Storage
- Printing → Printing

Therefore, the missing values can be reliably restored from the established product naming convention rather than assigned an arbitrary category.

In [11]:
# Define validated product-name-group mappings

subcategory_mapping = {
    "Cookware": "Cookware",
    "Storage": "Storage",
    "Printing": "Printing"
}


missing_before = (
    products_clean["subcategory"]
    .isna()
    .sum()
)


# Fill only currently missing subcategories

missing_mask = products_clean["subcategory"].isna()

products_clean.loc[
    missing_mask,
    "subcategory"
] = (
    products_clean.loc[
        missing_mask,
        "product_name_group"
    ]
    .map(subcategory_mapping)
)


missing_after = (
    products_clean["subcategory"]
    .isna()
    .sum()
)


print(f"Missing Before Cleaning : {missing_before:,}")
print(f"Values Restored         : {missing_before - missing_after:,}")
print(f"Missing After Cleaning  : {missing_after:,}")


print("\nCorrected Products:")

print(
    products_clean.loc[
        products_clean["product_id"].isin(
            ["SKU0184", "SKU0281", "SKU0424"]
        ),
        [
            "product_id",
            "product_name",
            "category",
            "subcategory"
        ]
    ]
    .to_string(index=False)
)

Missing Before Cleaning : 3
Values Restored         : 3
Missing After Cleaning  : 0

Corrected Products:
product_id          product_name        category subcategory
   SKU0184 Cookware Product 0184  Home & Kitchen    Cookware
   SKU0281  Storage Product 0281  Home & Kitchen     Storage
   SKU0424 Printing Product 0424 Office Supplies    Printing


## Step 12 — Evaluate Missing Supplier Quality Rating

One supplier has a missing quality rating.

The Data Quality Audit found no meaningful relationship between supplier quality rating and observed delivery-performance measures. Therefore, delivery metrics should not be used to infer the missing rating.

The existing quality-rating distribution is reviewed before selecting an appropriate statistical imputation strategy.

No value is modified in this step.

In [12]:
# Evaluate supplier quality rating distribution

quality_rating = suppliers_clean["quality_rating"]

print(f"Total Suppliers        : {len(suppliers_clean):,}")
print(f"Missing Ratings        : {quality_rating.isna().sum():,}")

print(
    f"Mean Quality Rating    : "
    f"{quality_rating.mean():.2f}"
)

print(
    f"Median Quality Rating  : "
    f"{quality_rating.median():.2f}"
)

print(
    f"Minimum Rating         : "
    f"{quality_rating.min():.2f}"
)

print(
    f"Maximum Rating         : "
    f"{quality_rating.max():.2f}"
)


print("\nQuality Rating Summary:")

print(
    quality_rating
    .describe()
    .round(2)
    .to_string()
)


print("\nSupplier With Missing Quality Rating:")

print(
    suppliers_clean.loc[
        suppliers_clean["quality_rating"].isna()
    ]
    .to_string(index=False)
)

Total Suppliers        : 100
Missing Ratings        : 1
Mean Quality Rating    : 4.05
Median Quality Rating  : 4.03
Minimum Rating         : 3.14
Maximum Rating         : 4.99

Quality Rating Summary:
count    99.00
mean      4.05
std       0.55
min       3.14
25%       3.60
50%       4.03
75%       4.53
max       4.99

Supplier With Missing Quality Rating:
supplier_id supplier_name supplier_region  contract_lead_time_days  quality_rating preferred_supplier_flag  reliability_factor
     SUP051  Supplier 051            West                       22             NaN                       N               0.808


## Step 13 — Handle Missing Supplier Quality Rating

The supplier quality-rating distribution was evaluated before imputation.

Only one supplier has a missing rating. The observed mean (4.05) and median (4.03) are very similar, indicating no material difference between the two central estimates.

Because delivery-performance variables were previously shown to have no meaningful relationship with quality rating, they are not used for imputation.

The missing quality rating is therefore replaced with the median observed supplier quality rating of 4.03.

In [13]:
# Handle missing supplier quality rating

missing_before = (
    suppliers_clean["quality_rating"]
    .isna()
    .sum()
)


median_quality_rating = (
    suppliers_clean["quality_rating"]
    .median()
)


suppliers_clean["quality_rating"] = (
    suppliers_clean["quality_rating"]
    .fillna(median_quality_rating)
)


missing_after = (
    suppliers_clean["quality_rating"]
    .isna()
    .sum()
)


print(f"Missing Before Cleaning : {missing_before:,}")

print(
    f"Median Used             : "
    f"{median_quality_rating:.2f}"
)

print(
    f"Values Imputed          : "
    f"{missing_before - missing_after:,}"
)

print(f"Missing After Cleaning  : {missing_after:,}")


print("\nSUP051 After Cleaning:")

print(
    suppliers_clean.loc[
        suppliers_clean["supplier_id"] == "SUP051",
        [
            "supplier_id",
            "supplier_name",
            "supplier_region",
            "quality_rating",
            "reliability_factor"
        ]
    ]
    .to_string(index=False)
)

Missing Before Cleaning : 1
Median Used             : 4.03
Values Imputed          : 1
Missing After Cleaning  : 0

SUP051 After Cleaning:
supplier_id supplier_name supplier_region  quality_rating  reliability_factor
     SUP051  Supplier 051            West            4.03               0.808


## Step 14 — Flag Invalid Shipment Chronology

The Data Quality Audit identified shipment records where `actual_delivery_date` occurs before `shipment_date`.

These records should not be silently deleted or have their dates arbitrarily modified because the correct shipment or delivery date cannot be reconstructed from the available data.

A data-quality flag is therefore created to identify chronology-valid and chronology-invalid shipment records.

This preserves the original records while allowing invalid chronology to be excluded from delivery-performance calculations.

In [14]:
# Flag shipment chronology validity

shipments_clean["chronology_valid_flag"] = np.where(
    shipments_clean["actual_delivery_date"]
    >= shipments_clean["shipment_date"],
    "Y",
    "N"
)


valid_count = (
    shipments_clean["chronology_valid_flag"]
    .eq("Y")
    .sum()
)

invalid_count = (
    shipments_clean["chronology_valid_flag"]
    .eq("N")
    .sum()
)


print(f"Total Shipment Records      : {len(shipments_clean):,}")
print(f"Valid Chronology Records    : {valid_count:,}")
print(f"Invalid Chronology Records  : {invalid_count:,}")

print(
    f"Invalid Chronology Rate     : "
    f"{invalid_count / len(shipments_clean) * 100:.2f}%"
)


print("\nChronology Flag Distribution:")

print(
    shipments_clean["chronology_valid_flag"]
    .value_counts()
    .to_string()
)

Total Shipment Records      : 24,798
Valid Chronology Records    : 20,331
Invalid Chronology Records  : 4,467
Invalid Chronology Rate     : 18.01%

Chronology Flag Distribution:
chronology_valid_flag
Y    20331
N     4467


## Step 15 — Create Chronology-Safe Shipment KPI Fields

Invalid shipment chronology can distort delivery-performance metrics even when the existing KPI calculations are mathematically consistent with the recorded dates.

To preserve data lineage:

- Original shipment KPI fields are retained unchanged.
- New cleaned KPI fields are created for analytical use.
- KPI values are retained only for chronology-valid shipments.
- Chronology-invalid shipments receive missing analytical KPI values.

This prevents the 4,467 invalid chronology records from overstating delivery performance while preserving the original source information.

In [15]:
# Create chronology-safe analytical shipment KPI fields

valid_chronology = (
    shipments_clean["chronology_valid_flag"] == "Y"
)


shipments_clean["delivery_delay_days_clean"] = (
    shipments_clean["delivery_delay_days"]
    .where(valid_chronology)
)


shipments_clean["on_time_flag_clean"] = (
    shipments_clean["on_time_flag"]
    .where(valid_chronology)
)


shipments_clean["otif_flag_clean"] = (
    shipments_clean["otif_flag"]
    .where(valid_chronology)
)


print("Chronology-Safe Shipment KPI Fields Created.\n")


print(
    f"Valid Delivery Delay Values   : "
    f"{shipments_clean['delivery_delay_days_clean'].notna().sum():,}"
)

print(
    f"Excluded Delivery Delay Values: "
    f"{shipments_clean['delivery_delay_days_clean'].isna().sum():,}"
)

print(
    f"Valid On-Time KPI Values      : "
    f"{shipments_clean['on_time_flag_clean'].notna().sum():,}"
)

print(
    f"Valid OTIF KPI Values         : "
    f"{shipments_clean['otif_flag_clean'].notna().sum():,}"
)


print("\nInvalid Chronology Records — Clean KPI Check:")

print(
    shipments_clean.loc[
        shipments_clean["chronology_valid_flag"] == "N",
        [
            "chronology_valid_flag",
            "delivery_delay_days_clean",
            "on_time_flag_clean",
            "otif_flag_clean"
        ]
    ]
    .isna()
    .sum()
    .to_string()
)

Chronology-Safe Shipment KPI Fields Created.

Valid Delivery Delay Values   : 20,331
Excluded Delivery Delay Values: 4,467
Valid On-Time KPI Values      : 20,331
Valid OTIF KPI Values         : 20,331

Invalid Chronology Records — Clean KPI Check:
chronology_valid_flag           0
delivery_delay_days_clean    4467
on_time_flag_clean           4467
otif_flag_clean              4467


## Step 16 — Remove Temporary Helper Columns

The temporary `product_name_group` field was created only to validate and restore missing product subcategories.

Since the missing subcategories have now been successfully restored, this helper field is no longer required and should not be included in the final cleaned Products dataset.

In [16]:
# Remove temporary product helper column

print(
    "Helper Column Exists Before :",
    "product_name_group" in products_clean.columns
)


products_clean.drop(
    columns=["product_name_group"],
    inplace=True
)


print(
    "Helper Column Exists After  :",
    "product_name_group" in products_clean.columns
)

print(
    f"Final Products Column Count  : "
    f"{products_clean.shape[1]}"
)

Helper Column Exists Before : True
Helper Column Exists After  : False
Final Products Column Count  : 10


## Step 17 — Post-Cleaning Missing-Value Validation

Recheck missing values across all cleaned datasets after applying the confirmed cleaning treatments.

The expected result is:

- No missing customer credit tiers
- No missing product subcategories
- No missing supplier quality ratings
- Structural missing values remain only for open Purchase Orders
- Clean shipment KPI fields contain intentional missing values for chronology-invalid shipments

These intentional missing values are preserved by design and are not considered unresolved cleaning errors.

In [17]:
# Post-cleaning missing-value validation

clean_tables = {
    "Customers": customers_clean,
    "Products": products_clean,
    "Sales Orders": sales_orders_clean,
    "Sales Order Items": sales_order_items_clean,
    "Inventory": inventory_clean,
    "Inventory Transactions": inventory_transactions_clean,
    "Warehouses": warehouses_clean,
    "Shipments": shipments_clean,
    "Suppliers": suppliers_clean,
    "Purchase Orders": purchase_orders_clean
}


missing_summary = []


for table_name, df in clean_tables.items():

    missing_counts = df.isna().sum()

    for column, count in missing_counts.items():

        if count > 0:

            missing_summary.append({
                "table": table_name,
                "column": column,
                "missing_count": count
            })


missing_summary = pd.DataFrame(missing_summary)


print("Remaining Missing Values After Cleaning:\n")


if len(missing_summary) > 0:

    print(
        missing_summary
        .to_string(index=False)
    )

else:

    print("No missing values detected.")

Remaining Missing Values After Cleaning:

          table                    column  missing_count
      Shipments delivery_delay_days_clean           4467
      Shipments        on_time_flag_clean           4467
      Shipments           otif_flag_clean           4467
Purchase Orders       actual_receipt_date             30
Purchase Orders       supplier_delay_days             30


## Step 18 — Post-Cleaning Duplicate and Key Validation

Validate duplicate records and primary/candidate key integrity after cleaning.

The expected result is:

- No exact duplicate rows in any cleaned dataset
- No duplicate primary keys
- No duplicate Inventory composite keys
- Inventory Transactions contain exactly 110,000 unique transaction IDs after deduplication


In [18]:
# Post-cleaning duplicate and key validation

duplicate_validation = pd.DataFrame([
    {
        "table": "Customers",
        "rows": len(customers_clean),
        "duplicate_rows": customers_clean.duplicated().sum(),
        "duplicate_keys": customers_clean[
            "customer_id"
        ].duplicated().sum()
    },

    {
        "table": "Products",
        "rows": len(products_clean),
        "duplicate_rows": products_clean.duplicated().sum(),
        "duplicate_keys": products_clean[
            "product_id"
        ].duplicated().sum()
    },

    {
        "table": "Sales Orders",
        "rows": len(sales_orders_clean),
        "duplicate_rows": sales_orders_clean.duplicated().sum(),
        "duplicate_keys": sales_orders_clean[
            "order_id"
        ].duplicated().sum()
    },

    {
        "table": "Sales Order Items",
        "rows": len(sales_order_items_clean),
        "duplicate_rows": sales_order_items_clean.duplicated().sum(),
        "duplicate_keys": sales_order_items_clean[
            "order_item_id"
        ].duplicated().sum()
    },

    {
        "table": "Inventory",
        "rows": len(inventory_clean),
        "duplicate_rows": inventory_clean.duplicated().sum(),
        "duplicate_keys": inventory_clean.duplicated(
            subset=["product_id", "warehouse_id"]
        ).sum()
    },

    {
        "table": "Inventory Transactions",
        "rows": len(inventory_transactions_clean),
        "duplicate_rows": inventory_transactions_clean.duplicated().sum(),
        "duplicate_keys": inventory_transactions_clean[
            "transaction_id"
        ].duplicated().sum()
    },

    {
        "table": "Warehouses",
        "rows": len(warehouses_clean),
        "duplicate_rows": warehouses_clean.duplicated().sum(),
        "duplicate_keys": warehouses_clean[
            "warehouse_id"
        ].duplicated().sum()
    },

    {
        "table": "Shipments",
        "rows": len(shipments_clean),
        "duplicate_rows": shipments_clean.duplicated().sum(),
        "duplicate_keys": shipments_clean[
            "shipment_id"
        ].duplicated().sum()
    },

    {
        "table": "Suppliers",
        "rows": len(suppliers_clean),
        "duplicate_rows": suppliers_clean.duplicated().sum(),
        "duplicate_keys": suppliers_clean[
            "supplier_id"
        ].duplicated().sum()
    },

    {
        "table": "Purchase Orders",
        "rows": len(purchase_orders_clean),
        "duplicate_rows": purchase_orders_clean.duplicated().sum(),
        "duplicate_keys": purchase_orders_clean[
            "po_id"
        ].duplicated().sum()
    }
])


print("Post-Cleaning Duplicate & Key Validation:\n")

print(
    duplicate_validation
    .to_string(index=False)
)


print(
    "\nTotal Duplicate Rows Remaining :",
    duplicate_validation["duplicate_rows"].sum()
)

print(
    "Total Duplicate Keys Remaining :",
    duplicate_validation["duplicate_keys"].sum()
)

Post-Cleaning Duplicate & Key Validation:

                 table   rows  duplicate_rows  duplicate_keys
             Customers   2500               0               0
              Products    500               0               0
          Sales Orders  25000               0               0
     Sales Order Items  61397               0               0
             Inventory   3000               0               0
Inventory Transactions 110000               0               0
            Warehouses      6               0               0
             Shipments  24798               0               0
             Suppliers    100               0               0
       Purchase Orders   8500               0               0

Total Duplicate Rows Remaining : 0
Total Duplicate Keys Remaining : 0


## Step 19 — Post-Cleaning Foreign Key Integrity Validation

Revalidate the major relationships between cleaned datasets.

Cleaning should not introduce orphan records or break existing relationships between customer, product, order, inventory, shipment, supplier, warehouse, and procurement tables.

All foreign-key orphan counts are expected to remain zero.

In [19]:
# Post-cleaning foreign key integrity validation

fk_checks = [
    (
        "Sales Orders → Customers",
        sales_orders_clean["customer_id"],
        customers_clean["customer_id"]
    ),

    (
        "Sales Order Items → Sales Orders",
        sales_order_items_clean["order_id"],
        sales_orders_clean["order_id"]
    ),

    (
        "Sales Order Items → Products",
        sales_order_items_clean["product_id"],
        products_clean["product_id"]
    ),

    (
        "Sales Orders → Warehouses",
        sales_orders_clean["warehouse_id"],
        warehouses_clean["warehouse_id"]
    ),

    (
        "Inventory → Products",
        inventory_clean["product_id"],
        products_clean["product_id"]
    ),

    (
        "Inventory → Warehouses",
        inventory_clean["warehouse_id"],
        warehouses_clean["warehouse_id"]
    ),

    (
        "Inventory Transactions → Products",
        inventory_transactions_clean["product_id"],
        products_clean["product_id"]
    ),

    (
        "Inventory Transactions → Warehouses",
        inventory_transactions_clean["warehouse_id"],
        warehouses_clean["warehouse_id"]
    ),

    (
        "Shipments → Sales Orders",
        shipments_clean["order_id"],
        sales_orders_clean["order_id"]
    ),

    (
        "Purchase Orders → Products",
        purchase_orders_clean["product_id"],
        products_clean["product_id"]
    ),

    (
        "Purchase Orders → Suppliers",
        purchase_orders_clean["supplier_id"],
        suppliers_clean["supplier_id"]
    ),

    (
        "Purchase Orders → Warehouses",
        purchase_orders_clean["warehouse_id"],
        warehouses_clean["warehouse_id"]
    )
]


fk_results = []


for relationship, child_key, parent_key in fk_checks:

    orphan_count = (
        ~child_key.isin(parent_key)
    ).sum()

    fk_results.append({
        "relationship": relationship,
        "orphan_records": orphan_count
    })


fk_validation = pd.DataFrame(fk_results)


print("Post-Cleaning Foreign Key Validation:\n")

print(
    fk_validation
    .to_string(index=False)
)


print(
    "\nTotal Orphan Records :",
    fk_validation["orphan_records"].sum()
)

Post-Cleaning Foreign Key Validation:

                       relationship  orphan_records
           Sales Orders → Customers               0
   Sales Order Items → Sales Orders               0
       Sales Order Items → Products               0
          Sales Orders → Warehouses               0
               Inventory → Products               0
             Inventory → Warehouses               0
  Inventory Transactions → Products               0
Inventory Transactions → Warehouses               0
           Shipments → Sales Orders               0
         Purchase Orders → Products               0
        Purchase Orders → Suppliers               0
       Purchase Orders → Warehouses               0

Total Orphan Records : 0


## Step 20 — Validate Clean Shipment KPI Treatment

Validate the analytical shipment KPI fields created during cleaning.

The validation confirms that:

- Chronology-invalid shipments are excluded from clean KPI calculations.
- Chronology-valid shipments retain their original KPI values.
- Clean delivery-performance metrics are calculated only from valid shipment chronology.

In [20]:
# Validate cleaned shipment KPI treatment

valid_shipments = shipments_clean[
    shipments_clean["chronology_valid_flag"] == "Y"
]


invalid_shipments = shipments_clean[
    shipments_clean["chronology_valid_flag"] == "N"
]


on_time_rate_clean = (
    valid_shipments["on_time_flag_clean"]
    .eq("Y")
    .mean()
    * 100
)


otif_rate_clean = (
    valid_shipments["otif_flag_clean"]
    .eq("Y")
    .mean()
    * 100
)


avg_delay_clean = (
    valid_shipments["delivery_delay_days_clean"]
    .mean()
)


print(
    f"Chronology-Valid Shipments   : "
    f"{len(valid_shipments):,}"
)

print(
    f"Chronology-Invalid Shipments : "
    f"{len(invalid_shipments):,}"
)

print(
    f"Clean On-Time Delivery Rate  : "
    f"{on_time_rate_clean:.2f}%"
)

print(
    f"Clean OTIF Rate              : "
    f"{otif_rate_clean:.2f}%"
)

print(
    f"Clean Avg Delivery Delay     : "
    f"{avg_delay_clean:.2f} days"
)


print("\nInvalid Records With Non-Null Clean KPIs:")

print(
    "Delivery Delay :",
    invalid_shipments[
        "delivery_delay_days_clean"
    ].notna().sum()
)

print(
    "On-Time Flag   :",
    invalid_shipments[
        "on_time_flag_clean"
    ].notna().sum()
)

print(
    "OTIF Flag      :",
    invalid_shipments[
        "otif_flag_clean"
    ].notna().sum()
)

Chronology-Valid Shipments   : 20,331
Chronology-Invalid Shipments : 4,467
Clean On-Time Delivery Rate  : 71.22%
Clean OTIF Rate              : 46.25%
Clean Avg Delivery Delay     : 0.93 days

Invalid Records With Non-Null Clean KPIs:
Delivery Delay : 0
On-Time Flag   : 0
OTIF Flag      : 0


## Step 21 — Validate Cleaned Fields and Date Data Types

Perform a final validation of the directly cleaned master-data fields and converted date columns.

The validation confirms that:

- Customer credit tier contains no missing values.
- Product subcategory contains no missing values.
- Supplier quality rating contains no missing values.
- All validated date fields remain stored as datetime data types.
- Structural Purchase Order missing dates remain valid `NaT` values.


In [21]:
# Validate cleaned master-data fields

print("Cleaned Master-Data Validation:\n")

print(
    f"Missing Customer Credit Tier     : "
    f"{customers_clean['credit_tier'].isna().sum():,}"
)

print(
    f"Unknown Customer Credit Tier     : "
    f"{customers_clean['credit_tier'].eq('Unknown').sum():,}"
)

print(
    f"Missing Product Subcategory      : "
    f"{products_clean['subcategory'].isna().sum():,}"
)

print(
    f"Missing Supplier Quality Rating  : "
    f"{suppliers_clean['quality_rating'].isna().sum():,}"
)


# Validate date data types

date_validation = []

for table_name, (df, columns) in date_columns.items():

    for column in columns:

        date_validation.append({
            "table": table_name,
            "column": column,
            "dtype": str(df[column].dtype),
            "is_datetime": pd.api.types.is_datetime64_any_dtype(
                df[column]
            )
        })


date_validation = pd.DataFrame(date_validation)


print("\nDate Data Type Validation:\n")

print(
    date_validation
    .to_string(index=False)
)


print(
    "\nNon-Datetime Date Fields :",
    (~date_validation["is_datetime"]).sum()
)


print(
    "\nOpen PO Missing Actual Receipt Dates :",
    purchase_orders_clean[
        "actual_receipt_date"
    ].isna().sum()
)

Cleaned Master-Data Validation:

Missing Customer Credit Tier     : 0
Unknown Customer Credit Tier     : 20
Missing Product Subcategory      : 0
Missing Supplier Quality Rating  : 0

Date Data Type Validation:

                 table                  column          dtype  is_datetime
             Customers          customer_since datetime64[us]         True
          Sales Orders              order_date datetime64[us]         True
          Sales Orders requested_delivery_date datetime64[us]         True
     Sales Order Items              order_date datetime64[us]         True
Inventory Transactions        transaction_date datetime64[us]         True
             Shipments           shipment_date datetime64[us]         True
             Shipments requested_delivery_date datetime64[us]         True
             Shipments    actual_delivery_date datetime64[us]         True
       Purchase Orders                 po_date datetime64[us]         True
       Purchase Orders           expect

## Step 22 — Final Raw vs Clean Comparison

Compare the original raw datasets with their cleaned versions to document the impact of the cleaning process.

This comparison provides a final audit trail showing:

- Original and cleaned row counts
- Rows removed during cleaning
- Original and cleaned column counts
- Columns added for analytical data-quality handling

Only confirmed cleaning actions should produce structural differences.

In [22]:
# Final raw vs clean comparison

raw_clean_pairs = [
    ("Customers", customers, customers_clean),
    ("Products", products, products_clean),
    ("Sales Orders", sales_orders, sales_orders_clean),
    ("Sales Order Items", sales_order_items, sales_order_items_clean),
    ("Inventory", inventory, inventory_clean),
    (
        "Inventory Transactions",
        inventory_transactions,
        inventory_transactions_clean
    ),
    ("Warehouses", warehouses, warehouses_clean),
    ("Shipments", shipments, shipments_clean),
    ("Suppliers", suppliers, suppliers_clean),
    ("Purchase Orders", purchase_orders, purchase_orders_clean)
]


comparison_results = []


for table_name, raw_df, clean_df in raw_clean_pairs:

    comparison_results.append({
        "table": table_name,
        "raw_rows": len(raw_df),
        "clean_rows": len(clean_df),
        "rows_removed": len(raw_df) - len(clean_df),
        "raw_columns": raw_df.shape[1],
        "clean_columns": clean_df.shape[1],
        "columns_added": clean_df.shape[1] - raw_df.shape[1]
    })


raw_vs_clean = pd.DataFrame(comparison_results)


print("Final Raw vs Clean Dataset Comparison:\n")

print(
    raw_vs_clean
    .to_string(index=False)
)


print(
    "\nTotal Rows Removed :",
    raw_vs_clean["rows_removed"].sum()
)

print(
    "Net Columns Added :",
    raw_vs_clean["columns_added"].sum()
)

Final Raw vs Clean Dataset Comparison:

                 table  raw_rows  clean_rows  rows_removed  raw_columns  clean_columns  columns_added
             Customers      2500        2500             0            6              6              0
              Products       500         500             0           10             10              0
          Sales Orders     25000       25000             0           14             14              0
     Sales Order Items     61397       61397             0           16             16              0
             Inventory      3000        3000             0           11             11              0
Inventory Transactions    110045      110000            45            6              6              0
            Warehouses         6           6             0            5              5              0
             Shipments     24798       24798             0           10             14              4
             Suppliers       100         1

## Step 23 — Export Cleaned Datasets

Export all validated cleaned datasets to the processed-data directory.

The original raw CSV files remain unchanged.

These processed datasets will serve as the clean data layer for downstream SQL analysis, exploratory analysis, business analysis, and dashboard development.

In [23]:
# Export cleaned datasets

clean_exports = {
    "customers_clean.csv": customers_clean,
    "products_clean.csv": products_clean,
    "sales_orders_clean.csv": sales_orders_clean,
    "sales_order_items_clean.csv": sales_order_items_clean,
    "inventory_clean.csv": inventory_clean,
    "inventory_transactions_clean.csv": inventory_transactions_clean,
    "warehouses_clean.csv": warehouses_clean,
    "shipments_clean.csv": shipments_clean,
    "suppliers_clean.csv": suppliers_clean,
    "purchase_orders_clean.csv": purchase_orders_clean
}


for filename, df in clean_exports.items():

    output_file = processed_data_path / filename

    df.to_csv(
        output_file,
        index=False,
        date_format="%Y-%m-%d"
    )

    print(
        f"{filename:<35} "
        f"{df.shape[0]:>8,} rows × "
        f"{df.shape[1]:>2} columns"
    )


print(
    f"\nCleaned datasets exported to:\n"
    f"{processed_data_path}"
)

customers_clean.csv                    2,500 rows ×  6 columns
products_clean.csv                       500 rows × 10 columns
sales_orders_clean.csv                25,000 rows × 14 columns
sales_order_items_clean.csv           61,397 rows × 16 columns
inventory_clean.csv                    3,000 rows × 11 columns
inventory_transactions_clean.csv     110,000 rows ×  6 columns
warehouses_clean.csv                       6 rows ×  5 columns
shipments_clean.csv                   24,798 rows × 14 columns
suppliers_clean.csv                      100 rows ×  7 columns
purchase_orders_clean.csv              8,500 rows × 14 columns

Cleaned datasets exported to:
c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\processed


## Step 24 — Final Export Verification

Verify that all cleaned datasets were successfully exported and can be reloaded from the processed-data directory.

The exported files are compared with their in-memory cleaned DataFrames to confirm:

- All expected files exist
- Exported row counts match
- Exported column counts match
- No records or columns were accidentally lost during export

Successful completion of this validation confirms that the cleaned data layer is ready for downstream analysis.

In [24]:
# Final export verification

export_validation = []


for filename, original_df in clean_exports.items():

    file_path = processed_data_path / filename

    file_exists = file_path.exists()

    if file_exists:

        exported_df = pd.read_csv(file_path)

        exported_rows = len(exported_df)
        exported_columns = exported_df.shape[1]

    else:

        exported_rows = None
        exported_columns = None


    export_validation.append({
        "file": filename,
        "exists": file_exists,
        "expected_rows": len(original_df),
        "exported_rows": exported_rows,
        "expected_columns": original_df.shape[1],
        "exported_columns": exported_columns
    })


export_validation = pd.DataFrame(export_validation)


export_validation["rows_match"] = (
    export_validation["expected_rows"]
    == export_validation["exported_rows"]
)


export_validation["columns_match"] = (
    export_validation["expected_columns"]
    == export_validation["exported_columns"]
)


print("Final Export Verification:\n")

print(
    export_validation
    .to_string(index=False)
)


all_valid = (
    export_validation["exists"].all()
    and export_validation["rows_match"].all()
    and export_validation["columns_match"].all()
)


print("\nAll Files Exist       :", export_validation["exists"].all())
print("All Row Counts Match  :", export_validation["rows_match"].all())
print("All Column Counts Match:", export_validation["columns_match"].all())

print("\nFINAL EXPORT STATUS   :", "PASSED" if all_valid else "FAILED")

Final Export Verification:

                            file  exists  expected_rows  exported_rows  expected_columns  exported_columns  rows_match  columns_match
             customers_clean.csv    True           2500           2500                 6                 6        True           True
              products_clean.csv    True            500            500                10                10        True           True
          sales_orders_clean.csv    True          25000          25000                14                14        True           True
     sales_order_items_clean.csv    True          61397          61397                16                16        True           True
             inventory_clean.csv    True           3000           3000                11                11        True           True
inventory_transactions_clean.csv    True         110000         110000                 6                 6        True           True
            warehouses_clean.csv  